# 📓 Chapter 2 — Practice Notebook
## Tokens and Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

### How to use this notebook
- Run cells top to bottom with `Shift + Enter`
- `# 🏋️ EXERCISE` = complete it yourself first
- `# ✅ SOLUTION` = check after you've tried

### What you will practice
1. Tokenization — encode, inspect, and decode tokens
2. Compare tokenizers across different models
3. Contextualized token embeddings
4. Sentence embeddings + semantic search
5. word2vec — explore and build a song recommender
6. Knowledge check

---

In [ ]:
# 🛠️ Setup — run this first
!pip install transformers accelerate sentence-transformers gensim scikit-learn -q
print("✅ Setup complete!")

---
## Part 1 — Tokenization

Tokenization is the first step in every LLM pipeline. Text goes in, token IDs come out.
Let's make this concrete by inspecting exactly how a tokenizer works.

In [ ]:
# ── DEMO: Load a tokenizer and inspect tokens ────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

prompt = "Write an email apologizing to Sarah for the tragic gardening mishap."

# Encode: text → token IDs
input_ids = tokenizer(prompt, return_tensors="pt").input_ids
print("Token IDs:")
print(input_ids)
print(f"\nTotal tokens: {input_ids.shape[1]}")
print(f"Total characters: {len(prompt)}")
print(f"Ratio: {input_ids.shape[1] / len(prompt.split()):.2f} tokens per word")

In [ ]:
# ── DEMO: Inspect each token ─────────────────────────────────────────────────
print(f"{'Token ID':<12} {'Token Text'}")
print("-" * 30)
for token_id in input_ids[0]:
    token_text = tokenizer.decode(token_id)
    print(f"{token_id.item():<12} '{token_text}'")

In [ ]:
# 🏋️ EXERCISE 1 — Count Tokens in Different Text Types
# ─────────────────────────────────────────────────────────────────────────────
# Different types of text tokenize very differently.
# Use the tokenizer to count tokens for each of these.
# Then answer: which is most/least token-efficient?

texts = {
    "Simple English": "The cat sat on the mat.",
    "Technical jargon": "Hyperparametrization of the stochastic gradient descent optimizer.",
    "Python code": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
    "Emoji": "🎉🚀💻🤖🌟",
    "Numbers": "1234567890 3.14159 $99.99 2024-01-15",
}

# YOUR CODE HERE — tokenize each and print token counts


In [ ]:
# ✅ SOLUTION
print(f"{'Category':<25} {'Words':<10} {'Tokens':<10} {'Tokens/Word'}")
print("-" * 60)
for category, text in texts.items():
    words = len(text.split())
    tokens = len(tokenizer(text).input_ids)
    ratio = tokens / max(words, 1)
    print(f"{category:<25} {words:<10} {tokens:<10} {ratio:.2f}")

print("\n💡 INSIGHTS:")
print("   - Emojis are expensive: each emoji = multiple tokens (byte fallback)")
print("   - Code can be efficient IF the tokenizer is code-optimized")
print("   - Simple English ≈ 1.3 tokens/word (the common rule of thumb)")
print("   - Technical jargon often splits into more subword tokens")

In [ ]:
# 🏋️ EXERCISE 2 — Compare BERT vs GPT Tokenizers
# ─────────────────────────────────────────────────────────────────────────────
# BERT and GPT use different tokenization algorithms (WordPiece vs BPE).
# Let's see how they handle the same text differently.
#
# Tasks:
#   1. Load BERT tokenizer: "bert-base-uncased"
#   2. Tokenize this text with both BERT and Phi-3 tokenizers:
#      text = "CAPITALIZATION and emojis 🎵 and Python code: elif True:"
#   3. Print each token for both tokenizers side by side
#   4. Note 3 differences you observe

text = "CAPITALIZATION and emojis 🎵 and Python code: elif True:"

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
phi3_tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

bert_ids = bert_tokenizer(text).input_ids
phi3_ids = phi3_tokenizer(text).input_ids

bert_tokens = [bert_tokenizer.decode(t) for t in bert_ids]
phi3_tokens = [phi3_tokenizer.decode(t) for t in phi3_ids]

print(f"BERT tokens ({len(bert_ids)}):")
print(" | ".join(bert_tokens))
print()
print(f"Phi-3 tokens ({len(phi3_ids)}):")
print(" | ".join(phi3_tokens))
print()
print("💡 DIFFERENCES to notice:")
print("   1. BERT lowercases everything — Phi-3 preserves capitalization")
print("   2. BERT turns emoji into [UNK] — Phi-3 encodes via byte fallback")
print("   3. BERT may split 'elif' differently than Phi-3")
print("   4. BERT wraps with [CLS] and [SEP] — Phi-3 uses <s>")

---
## Part 2 — Contextualized Token Embeddings

A tokenizer gives us token IDs → the embedding matrix gives us static vectors → Transformer layers give us contextualized vectors. Let's see how the same word gets different representations in different contexts.

In [ ]:
# ── DEMO: Contextualized Embeddings ──────────────────────────────────────────
from transformers import AutoModel, AutoTokenizer
import torch

model_name = "microsoft/deberta-v3-xsmall"
ctx_tokenizer = AutoTokenizer.from_pretrained(model_name)
ctx_model = AutoModel.from_pretrained(model_name)
ctx_model.eval()

def get_token_embedding(sentence, target_word):
    """Get the contextualized embedding for a specific word in a sentence."""
    tokens = ctx_tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        output = ctx_model(**tokens)[0]
    
    # Find which token corresponds to target_word
    token_strings = [ctx_tokenizer.decode(t) for t in tokens['input_ids'][0]]
    
    for i, tok in enumerate(token_strings):
        if target_word.lower() in tok.lower():
            return output[0][i].detach().numpy(), i, token_strings
    return None, None, token_strings

# Two sentences with very different meanings of "bank"
sentence1 = "I went to the bank to deposit my salary."
sentence2 = "We sat on the bank of the river and watched the sunset."

emb1, idx1, tokens1 = get_token_embedding(sentence1, "bank")
emb2, idx2, tokens2 = get_token_embedding(sentence2, "bank")

print(f"Sentence 1 tokens: {tokens1}")
print(f"Sentence 2 tokens: {tokens2}")
print(f"\n'bank' embedding dimension: {len(emb1)}")

In [ ]:
# ── DEMO: Compare the two 'bank' embeddings ───────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

sim = cosine_similarity([emb1], [emb2])[0][0]

print(f"Cosine similarity between the two 'bank' embeddings: {sim:.4f}")
print()
if sim > 0.95:
    print("⚠️  Very similar — model might not distinguish contexts well")
elif sim > 0.80:
    print("👍 Fairly different — model captures some context")
else:
    print("✅ Very different — model strongly distinguishes contexts!")

print()
print("💡 For comparison: in word2vec (static embeddings), these would be")
print("   IDENTICAL (similarity = 1.0) because 'bank' always maps to the same vector.")
print("   Contextualized embeddings solve this problem.")

In [ ]:
# 🏋️ EXERCISE 3 — Find Another Ambiguous Word
# ─────────────────────────────────────────────────────────────────────────────
# Find a word that means different things in different contexts.
# Write two sentences with clearly different meanings of that word.
# Compute the similarity between the contextualized embeddings.
#
# Suggestions: "bat", "light", "fly", "right", "spring", "match", "plant"

word = "bat"   # change this
sent_a = "The cricket player picked up his bat and prepared to face the bowler."
sent_b = "A bat flew out of the cave at dusk, hunting for insects."

# YOUR CODE HERE


---
## Part 3 — Sentence Embeddings and Semantic Search

For most app-building tasks, you don't need per-token embeddings — you need a single vector per sentence. That's what sentence embedding models give you. Let's build a real semantic search system.

In [ ]:
# ── DEMO: Sentence Embeddings ─────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "The dog runs fast in the park.",
    "A puppy is sprinting quickly through the garden.",   # similar to sentence 1
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",  # similar to sentence 3
    "I love eating pizza on Friday nights.",
]

embeddings = embed_model.encode(sentences)
print(f"Each sentence → vector of {embeddings.shape[1]} numbers")
print(f"Total: {embeddings.shape[0]} sentences × {embeddings.shape[1]} dims")
print()

# Compute similarity matrix
sim_matrix = cosine_similarity(embeddings)

print("Similarity matrix (rows and columns = each sentence):")
for i, sent in enumerate(sentences):
    print(f"\n[{i}] {sent[:50]}...")
    for j, other in enumerate(sentences):
        if i != j:
            print(f"     vs [{j}]: {sim_matrix[i][j]:.3f}")

In [ ]:
# 🏋️ EXERCISE 4 — Build a Semantic FAQ Search
# ─────────────────────────────────────────────────────────────────────────────
# Scenario: You're building a help center search for an e-commerce app.
#
# Your FAQ articles:
faq_articles = [
    "How to track your order status and delivery",
    "Return and refund policy — how to send items back",
    "Payment methods accepted: credit card, PayPal, crypto",
    "How to create and manage your account",
    "Contacting customer support and response times",
    "Changing or cancelling an order before shipment",
    "Product warranty and repair services",
    "Shipping costs and international delivery options",
]
#
# User queries (notice: users DON'T use the exact same words as the article titles):
user_queries = [
    "where is my package?",
    "I want my money back",
    "can I pay with bitcoin?",
    "I need to speak to a human",
    "I changed my mind about my purchase",
]
#
# Tasks:
#   1. Embed all FAQ articles
#   2. For each user query: embed it, find the most similar FAQ article
#   3. Print: query → best matching article + similarity score
#
# Expected: queries should match the right articles even without exact word matches

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION

faq_embeddings = embed_model.encode(faq_articles)

print("Semantic FAQ Search Results:")
print("=" * 70)

for query in user_queries:
    query_embedding = embed_model.encode([query])
    similarities = cosine_similarity(query_embedding, faq_embeddings)[0]
    best_idx = np.argmax(similarities)
    best_score = similarities[best_idx]
    
    print(f"\n🔍 Query:  '{query}'")
    print(f"📄 Result: '{faq_articles[best_idx]}'")
    print(f"   Score:  {best_score:.3f}")

print()
print("💡 Notice: none of the user queries use the exact words from the FAQ titles.")
print("   'where is my package' → correctly finds 'How to track your order'")
print("   'I want my money back' → correctly finds 'Return and refund policy'")
print("   This is semantic search — and this is the foundation of RAG (Chapter 8).")

---
## Part 4 — word2vec and Embeddings for Recommendation

The same idea that makes word2vec work (words appearing together → similar vectors) can be applied to songs in playlists, products in carts, or movies in watch sessions.

In [ ]:
# ── DEMO: Pretrained word2vec — explore semantic relationships ────────────────
import gensim.downloader as api

print("Downloading GloVe embeddings (66MB, trained on Wikipedia)...")
word_model = api.load("glove-wiki-gigaword-50")
print("✅ Loaded!")

# Find nearest neighbors of 'king'
print("\nNearest neighbors of 'king':")
for word, score in word_model.most_similar([word_model['king']], topn=8):
    bar = "█" * int(score * 20)
    print(f"  {word:<15} {score:.3f}  {bar}")

In [ ]:
# 🏋️ EXERCISE 5 — The Famous word2vec Arithmetic
# ─────────────────────────────────────────────────────────────────────────────
# word2vec embeddings support vector arithmetic:
#   king - man + woman ≈ queen
#
# Try these analogies and see how well they work:
#   1. king - man + woman = ?
#   2. paris - france + italy = ?
#   3. programmer - computer + paintbrush = ?
#
# Hint: use word_model.most_similar(positive=[...], negative=[...], topn=3)

# YOUR CODE HERE


In [ ]:
# ✅ SOLUTION

analogies = [
    {"positive": ["king", "woman"], "negative": ["man"], "description": "king - man + woman"},
    {"positive": ["paris", "italy"], "negative": ["france"], "description": "paris - france + italy"},
    {"positive": ["programmer", "paintbrush"], "negative": ["computer"], "description": "programmer - computer + paintbrush"},
]

for analogy in analogies:
    results = word_model.most_similar(
        positive=analogy["positive"],
        negative=analogy["negative"],
        topn=3
    )
    top_words = [f"{w} ({s:.3f})" for w, s in results]
    print(f"{analogy['description']} = {top_words}")

print()
print("💡 The king→queen analogy typically works well.")
print("   Paris→Rome often works too (capital city relationship).")
print("   The programmer one is more creative — results vary!")
print("   This shows that embeddings capture semantic relationships mathematically.")

In [ ]:
# ── DEMO: Build a Song Recommender using word2vec ─────────────────────────────
# Treat songs as words, playlists as sentences → train word2vec → find similar songs

import pandas as pd
from urllib import request
from gensim.models import Word2Vec
import numpy as np

print("Downloading playlist dataset...")
try:
    data = request.urlopen(
        'https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt',
        timeout=30
    )
    lines = data.read().decode("utf-8").split('\n')[2:]
    playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

    songs_file = request.urlopen(
        'https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt',
        timeout=30
    )
    songs_file = songs_file.read().decode("utf-8").split('\n')
    songs = [s.rstrip().split('\t') for s in songs_file]
    songs_df = pd.DataFrame(data=songs, columns=['id', 'title', 'artist'])
    songs_df = songs_df.set_index('id')

    print(f"✅ Loaded {len(playlists)} playlists")
    print(f"   Example playlist: {playlists[0][:5]}...")

    # Train word2vec on playlists
    # songs = words, playlists = sentences
    print("\nTraining word2vec on playlists...")
    song_model = Word2Vec(
        playlists,
        vector_size=32,   # 32-dimensional embeddings per song
        window=20,        # look 20 songs left/right in each playlist
        negative=50,      # 50 negative examples per positive pair
        min_count=1,
        workers=4
    )
    print("✅ Model trained!")
    
    dataset_loaded = True
except Exception as e:
    print(f"⚠️  Could not download dataset: {e}")
    print("   This is expected if running offline. The concepts still apply.")
    dataset_loaded = False

In [ ]:
# ── DEMO: Get Song Recommendations ───────────────────────────────────────────

if dataset_loaded:
    def get_recommendations(song_id, topn=5):
        similar = song_model.wv.most_similar(positive=str(song_id), topn=topn)
        similar_ids = [s[0] for s in similar]
        print(f"\nSeed song: '{songs_df.loc[str(song_id), 'title']}' by {songs_df.loc[str(song_id), 'artist']}")
        print("Recommendations:")
        for sid in similar_ids:
            try:
                title = songs_df.loc[sid, 'title']
                artist = songs_df.loc[sid, 'artist']
                print(f"  - {title} — {artist}")
            except:
                print(f"  - Song ID {sid}")

    # Michael Jackson — Billie Jean
    get_recommendations(3822)

    # Metallica — Fade to Black
    get_recommendations(2172)
else:
    print("Dataset not loaded — showing conceptual explanation instead.")
    print()
    print("The word2vec algorithm on playlists works like this:")
    print("  Songs = words")
    print("  Playlists = sentences")
    print("  Songs that appear together in playlists → similar embeddings")
    print("  Find similar songs by computing cosine similarity between embeddings")

In [ ]:
# 🏋️ EXERCISE 6 — Think Like an Engineer
# ─────────────────────────────────────────────────────────────────────────────
# You work at an e-commerce company. You want to build a "you might also like"
# product recommendation system using the same word2vec approach.
#
# Answer these architecture questions as comments:

# Q1: What would your "words" and "sentences" be in this context?
# A:

# Q2: What data would you need to collect, and from where?
# A:

# Q3: What are the limitations of this approach?
#     (Think: cold start problem, new products, niche products)
# A:

# Q4: How would you serve recommendations in a real app?
#     (Think: pre-compute or on-demand? Where to store embeddings?)
# A:


In [ ]:
# ✅ SOLUTION

print("""Q1: Words and sentences:
  Words = product IDs
  Sentences = user purchase/browsing sessions (ordered sequence of products)
  A 'session' groups products a user interacted with in one visit.

Q2: Data needed:
  - User session logs from your analytics pipeline
  - Each session: list of product IDs in order of interaction
  - You need enough sessions per product for meaningful co-occurrence
  - Typically need 1M+ sessions for decent quality

Q3: Limitations:
  - Cold start: new products have no session data → no embedding → can't recommend
  - Popularity bias: popular products appear in many sessions → dominate recommendations
  - Niche products: rarely co-occur → poor embeddings
  - Temporal: doesn't capture seasonality or trend changes
  - Context-blind: doesn't know WHY a user bought something

Q4: Serving recommendations:
  - Pre-compute: calculate top-N similar products for each product offline
  - Store in a fast lookup store (Redis, DynamoDB) keyed by product_id
  - On page load: fetch recommendations for current product by product_id
  - For real-time personalization: store user embedding = average of
    recently viewed product embeddings, query vector DB for similar products
""")

---
## Part 5 — Knowledge Check

In [ ]:
# ❓ Q1: Why can't you use one model's tokenizer with a different model?
your_answer = """
...
"""
print(your_answer)

In [ ]:
# ❓ Q2: What is the difference between a static and a contextualized embedding?
#        Give a concrete example of why this matters.
your_answer = """
...
"""
print(your_answer)

In [ ]:
# ❓ Q3: PRACTICAL: Your app needs to process user-submitted documents in multiple
#        languages (English, Hindi, Arabic). What should you look for in a tokenizer?
your_answer = """
...
"""
print(your_answer)

In [ ]:
# ✅ MODEL ANSWERS

answers = {
    "Q1": """Each model has an embedding matrix with exactly one row per token in ITS
tokenizer's vocabulary. The embedding for token ID 14350 in Model A's matrix
corresponds to a completely different token in Model B's matrix — the IDs don't
transfer. Using a different tokenizer would pass wrong IDs to the model, producing
garbage output. The tokenizer and model are trained together and are inseparable.""",

    "Q2": """Static embeddings (word2vec): every occurrence of the word 'bank' gets
the same vector, regardless of context. 'river bank' and 'bank account' produce
identical vectors for 'bank' — the model can't distinguish meaning from context.

Contextualized embeddings (BERT, GPT): the same word 'bank' gets a different vector
depending on surrounding words. The Transformer's self-attention layers adjust each
token's representation based on the entire input sequence.

This matters because many words are ambiguous. NER (spotting that 'Apple' is a
company in a tech article but a fruit in a recipe) requires context-aware representations.""",

    "Q3": """Look for:
1. Multilingual tokenizer — trained on a dataset that includes Hindi and Arabic scripts.
   BERT-uncased is English-only (turns non-English to [UNK]). Use mBERT, XLM-R, or
   similar multilingual models instead.
2. Unicode/script support — tokenizer should handle Devanagari (Hindi) and Arabic
   script without falling back to [UNK].
3. Right-to-left support — Arabic is RTL; some tokenizers don't handle this well.
4. Byte-level fallback — ensures even unseen characters are representable.
Good choices: XLM-RoBERTa, mBERT (cased), or models specifically designed for
multilingual tasks."""
}

for q, ans in answers.items():
    print(f"{'='*65}")
    print(f"✅ {q}:")
    print(ans)
    print()

---
## 🎉 Chapter 2 Practice Complete!

### What you did:
- ✅ Encoded text into token IDs and decoded them back
- ✅ Compared how BERT and GPT tokenizers handle the same text
- ✅ Measured contextualized embeddings and proved they change with context
- ✅ Built a semantic FAQ search engine from scratch
- ✅ Explored word2vec vector arithmetic
- ✅ Built (or understood) a song recommendation system using word2vec
- ✅ Answered all knowledge check questions

### Before moving to Chapter 3:
Read `summary.md` — 10 minutes, locks in the whole chapter.

### What's next:
**Chapter 3 — Looking Inside Large Language Models**
You've seen tokens go IN to a model. Chapter 3 opens the black box and shows you exactly what happens inside — multi-head attention, feedforward layers, positional embeddings, and how the model generates the next token.